# Transfer Learning from ResNet50

In [1]:
import torch, torchvision
import numpy as np
from collect_data import collect_data, train_model

DATA_DIR = '../data/images'

# Load pretrained ResNet model
resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get training and validation data
train_loader, val_loader, test_loader = collect_data(DATA_DIR, preprocess)

1000 images processed
2000 images processed
3000 images processed
4000 images processed
5000 images processed
6000 images processed
7000 images processed
8000 images processed
9000 images processed
10000 images processed
11000 images processed
12000 images processed


In [2]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [3]:
class ResNetTransfer(torch.nn.Module):
    def __init__(self, resnetmodel):
        super(ResNetTransfer, self).__init__()
        # ResNet structure (without classifier)
        self.resnet = torch.nn.Sequential(resnetmodel.conv1,
                                            resnetmodel.bn1,
                                            resnetmodel.relu,
                                            resnetmodel.maxpool,
                                            resnetmodel.layer1,
                                            resnetmodel.layer2,
                                            resnetmodel.layer3,
                                            resnetmodel.layer4,
                                            resnetmodel.avgpool)
        # 2048 from the resnet convolution and 1 from the angles
        self.classifier = torch.nn.Sequential(torch.nn.Linear(2049, 256),
                                              torch.nn.Linear(256, 23))
    
    def forward(self, X, angles):
        r = self.resnet(X)
        f = torch.flatten(r, 1)
        f = torch.cat([f, angles], dim=1)
        y = self.classifier(f)
        return y

In [6]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [7]:
import copy

model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.resnet.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 530711


In [6]:
# Hyperparameters
epsilon = 1E-4

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=200, checkpoint_path='resnet_transfer_double_FULL.pth', graph_save_directory='./fully_trained_double')

Epoch [0/200], Train Loss: 2.91592, Val Loss: 2.87424 *, Train Acc: 0.12690, Validation Acc: 0.10333
Epoch [1/200], Train Loss: 2.65988, Val Loss: 2.75122 *, Train Acc: 0.19488, Validation Acc: 0.11111
Epoch [2/200], Train Loss: 2.52134, Val Loss: 2.71777 *, Train Acc: 0.20190, Validation Acc: 0.08667
Epoch [3/200], Train Loss: 2.42131, Val Loss: 2.71504 *, Train Acc: 0.22940, Validation Acc: 0.09111
Epoch [4/200], Train Loss: 2.36876, Val Loss: 2.67911 *, Train Acc: 0.23000, Validation Acc: 0.10167
Epoch [5/200], Train Loss: 2.31904, Val Loss: 2.65189 *, Train Acc: 0.25202, Validation Acc: 0.09944
Epoch [6/200], Train Loss: 2.27912, Val Loss: 2.62743 *, Train Acc: 0.26655, Validation Acc: 0.11167
Epoch [7/200], Train Loss: 2.26042, Val Loss: 2.69779, Train Acc: 0.27607, Validation Acc: 0.08333
Epoch [8/200], Train Loss: 2.24267, Val Loss: 2.67182, Train Acc: 0.27238, Validation Acc: 0.09778
Epoch [9/200], Train Loss: 2.18375, Val Loss: 2.59341 *, Train Acc: 0.29060, Validation Acc: 0.

Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-66c7deff-f05d-4287-9e02-015e2f274d91.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-66c7deff-f05d-4287-9e02-015e2f274d91.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-66c7deff-f05d-4287-9e02-015e2f274d91.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-66c7deff-f05d-4287-9e02-015e2f274d91.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-66c7deff-f05d-4287-9e02-015e2f274d91.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connec

In [8]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_transfer_double_FULL.pth', criterion, test_loader, device)

Loss: 2.591762901941935
Accuracy: 0.10833333333333334
Off-by-One Accuracy: 0.3288888888888889


In [10]:
# Save test data
torch.save(test_loader, "test_data_classifier.pt")

In [15]:
# Fine-tuning
finetune_model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)
finetune_model.load_state_dict(torch.load('resnet_transfer_double_FULL.pth', map_location=device))

for param in finetune_model.resnet.parameters():
    param.requires_grad = True

# Use different learning rate on different params to train resnet layers more slowly
finetune_optimizer = torch.optim.Adam([
    {'params': finetune_model.resnet.parameters(), 'lr': 1e-6},
    {'params': finetune_model.classifier.parameters(), 'lr': epsilon},
])

train_model(finetune_model, train_loader, val_loader, criterion, finetune_optimizer, device, num_epochs=200, checkpoint_path='resnet_finetuned_double_FULL.pth', graph_save_directory='./fully_finetuned_double')

Epoch [0/200], Train Loss: 2.03304, Val Loss: 2.60824 *, Train Acc: 0.33917, Validation Acc: 0.11722
Epoch [1/200], Train Loss: 1.91928, Val Loss: 2.54551 *, Train Acc: 0.36845, Validation Acc: 0.10722
Epoch [2/200], Train Loss: 1.83617, Val Loss: 2.48402 *, Train Acc: 0.39762, Validation Acc: 0.13278
Epoch [3/200], Train Loss: 1.74383, Val Loss: 2.50097, Train Acc: 0.42738, Validation Acc: 0.12389
Epoch [4/200], Train Loss: 1.67950, Val Loss: 2.49947, Train Acc: 0.44988, Validation Acc: 0.12333
Epoch [5/200], Train Loss: 1.62385, Val Loss: 2.49265, Train Acc: 0.46929, Validation Acc: 0.13278
Epoch [6/200], Train Loss: 1.56195, Val Loss: 2.52450, Train Acc: 0.48369, Validation Acc: 0.11556
Epoch [7/200], Train Loss: 1.49366, Val Loss: 2.53405, Train Acc: 0.51155, Validation Acc: 0.12889
Epoch [8/200], Train Loss: 1.45209, Val Loss: 2.52427, Train Acc: 0.52607, Validation Acc: 0.14278
Epoch [9/200], Train Loss: 1.37253, Val Loss: 2.55268, Train Acc: 0.55381, Validation Acc: 0.13778
Epoc

In [10]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_finetuned_double_FULL.pth', criterion, test_loader, device)

Loss: 2.2040289465586342
Accuracy: 0.24833333333333332
Off-by-One Accuracy: 0.44722222222222224
